# Regularization in PyTorch: fighting overfitting

_PyTorch (a complete course)_

**Dropout, batch norm, weight decay, early stopping, and grad clipping — the tools that keep a PyTorch model from memorizing the training set.**

Overfitting happens when a model has enough capacity to memorize the training set — it fits the noise, not just the signal. Every technique here is a different way to limit effective capacity or add training variety so memorization is harder than learning the real pattern:

       
         
- Dropout and batch norm add noise / constraints inside the network.
         
- Weight decay shrinks the weights, keeping the function smooth.
         
- Early stopping limits how long the model trains, so it never reaches the deeply-memorized state.
         
- Augmentation grows the apparent dataset so there is simply more to fit.
       

---

This notebook is a practice scaffold for the **Regularization in PyTorch: fighting overfitting** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import copy
import torch
import torch.nn as nn

torch.manual_seed(0)

# --- tiny synthetic problem: few samples, many noise features (easy to overfit) ---
N_TRAIN, N_VAL, D = 64, 400, 200
def make(n):
    X = torch.randn(n, D)
    logits = 1.5 * X[:, 0] - 1.2 * X[:, 1]        # only 2 features matter; rest is noise
    y = (torch.rand(n) < torch.sigmoid(logits)).long()
    return X, y
Xtr, ytr = make(N_TRAIN)
Xva, yva = make(N_VAL)

# --- a model WITH batch norm + dropout (regularizers live INSIDE the model) ---
model = nn.Sequential(
    nn.Linear(D, 64),
    nn.BatchNorm1d(64),     # train: batch stats; eval: running stats  (mode-dependent!)
    nn.ReLU(),
    nn.Dropout(0.5),        # train: drops 50% of units; eval: pass-through (mode-dependent!)
    nn.Linear(64, 2),       # 2 logits; CrossEntropyLoss wants RAW logits + class indices
)

loss_fn = nn.CrossEntropyLoss()
# AdamW = decoupled weight decay = the CORRECT way to do L2 with Adam.
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# --- training loop with EARLY STOPPING + GRADIENT CLIPPING ---
best_val = float("inf")
best_state = copy.deepcopy(model.state_dict())   # snapshot of the BEST weights
patience, since_best = 15, 0

for epoch in range(200):
    # ---- train ----
    model.train()                                 # dropout ON, batchnorm uses batch stats
    optimizer.zero_grad()                         # grads accumulate -> always zero first
    logits = model(Xtr)
    loss = loss_fn(logits, ytr)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # cap gradient size
    optimizer.step()

    # ---- validate ----
    model.eval()                                  # dropout OFF, batchnorm uses running stats
    with torch.no_grad():                         # no graph needed at eval -> save memory
        val_loss = loss_fn(model(Xva), yva).item()

    # ---- early stopping: keep the BEST weights, not the last ----
    if val_loss < best_val - 1e-4:
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())   # new best -> snapshot it
        since_best = 0
    else:
        since_best += 1
        if since_best >= patience:
            print(f"early stop at epoch {epoch} (best val loss {best_val:.4f})")
            break

model.load_state_dict(best_state)                 # RESTORE the best weights before using
model.eval()
with torch.no_grad():
    val_acc = (model(Xva).argmax(1) == yva).float().mean().item()
print(f"restored best val loss = {best_val:.4f} | val accuracy = {val_acc:.3f}")


## Visualize the data & results

_Does regularization actually stop overfitting? Train the same tiny logistic model on few samples with many noise features, WITHOUT vs WITH L2 (weight decay), and plot validation loss over epochs._

In [ ]:
import numpy as np
rng = np.random.RandomState(0)

# tiny REAL overfitting setup: very few samples, mostly noise features
n_train, n_val, d = 30, 300, 400
def make(n):
    X = rng.randn(n, d)
    logits = 1.5*X[:,0] - 1.2*X[:,1]      # only 2 informative features; rest is noise
    y = (rng.rand(n) < 1/(1+np.exp(-logits))).astype(float)
    return X, y
Xtr, ytr = make(n_train)
Xva, yva = make(n_val)

def sigmoid(z): return 1/(1+np.exp(-np.clip(z, -30, 30)))
def bce(p, y):
    p = np.clip(p, 1e-9, 1-1e-9)
    return -np.mean(y*np.log(p) + (1-y)*np.log(1-p))

def train(l2, epochs=300, lr=0.5):
    w = np.zeros(d); b = 0.0; val = []
    for _ in range(epochs):
        p = sigmoid(Xtr @ w + b)
        g = p - ytr
        gw = Xtr.T @ g / n_train + l2 * w     # the +l2*w term IS weight decay / L2
        gb = g.mean()
        w -= lr*gw; b -= lr*gb
        val.append(bce(sigmoid(Xva @ w + b), yva))
    return val

val_no = train(l2=0.0)     # no regularization -> overfits, val loss rises
val_l2 = train(l2=0.1)     # L2 weight decay   -> val loss stays low

idx = range(0, 300, 5)     # subsample to 60 points
print("no_reg:", [round(val_no[i], 3) for i in idx])
print("l2    :", [round(val_l2[i], 3) for i in idx])
print("no_reg start->final:", round(val_no[0],3), "->", round(val_no[-1],3))   # 0.812 -> 1.117
print("l2     start->final:", round(val_l2[0],3), "->", round(val_l2[-1],3))    # 0.812 -> 0.836


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your validation predictions change a little every time you run them on the same input, and they are worse than your training metrics suggest. The model has nn.Dropout(0.5) and nn.BatchNorm1d layers. What did you most likely forget, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the symptom: non-deterministic predictions on identical input. — _Only a training-mode layer injects randomness; at eval everything should be deterministic._
- Recall that dropout randomly zeros activations and batch norm uses batch statistics — but only while the module is in train() mode. — _These layers are mode-dependent; leaving them in train mode keeps the noise on._

**Answer:** You forgot model.eval() before evaluating. Dropout was still randomly zeroing units (hence the changing outputs) and batch norm was using batch statistics instead of its running averages. Fix: call model.eval() (and wrap inference in torch.no_grad()), then model.train() when you resume training.

</details>

**Problem 2.** You want true L2 regularization with the Adam family. Why is optim.Adam(params, weight_decay=1e-2) not quite what you want, and what should you use instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that classic L2 adds $\lambda \lVert w \rVert^2$ to the loss, so the decay term passes through Adam's per-parameter adaptive scaling. — _Adam rescales every gradient component, which distorts the intended uniform shrink._
- Recall that AdamW decouples the decay: it subtracts $\eta\,\lambda\,w$ directly, outside the adaptive update. — _That gives the clean weight-shrink the L2 penalty is supposed to produce._

**Answer:** With plain Adam the weight_decay gets entangled with the adaptive learning-rate scaling, so it is not the L2 you intended. Use optim.AdamW(params, lr=1e-3, weight_decay=1e-2), which applies decoupled weight decay correctly.

</details>

**Problem 3.** Your early-stopping loop stops 5 epochs after the best validation loss, then you save and deploy model as-is. Why might the deployed model be worse than the best one you saw, and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that you keep training (and updating weights) for the 5 patience epochs after the best point. — _Those extra steps move the weights past the best-generalizing state — validation loss was rising._
- Recall that you must snapshot model.state_dict() at each new best and reload it at the end. — _Otherwise the final in-memory weights are the worse, later ones._

**Answer:** The weights in memory are from 5 epochs after the best, where validation loss had already risen. Fix: save best_state = copy.deepcopy(model.state_dict()) whenever validation loss hits a new low, and model.load_state_dict(best_state) before saving / deploying — restore the BEST weights, not the last ones.

</details>